# Le Data Model Python

Les attributs et méthodes spéciaux (entourés de double underscore `__`) font partie du **data model** de Python. Ils permettent de personnaliser le comportement des objets.

---

## Attributs spéciaux (lecture seule)

Ces attributs sont accessibles sur la classe ou sur ses instances :

| Attribut | Disponible sur | Description |
|---|---|---|
| `__name__` | classe | Renvoie le nom de la classe |
| `__bases__` | classe | Renvoie un tuple des classes parentes |
| `__class__` | objet | Renvoie la classe à partir de laquelle l'objet a été créé |
| `__dict__` | objet | Renvoie un dictionnaire des attributs et méthodes |
| `__doc__` | classe / objet | Renvoie la docstring de la classe |

### Exemple d'utilisation des attributs spéciaux

In [ ]:
class MaClasse:
    """Classe d'exemple pour illustrer les attributs spéciaux."""

    def methode(self):
        pass

#### `__name__` — Nom de la classe

In [ ]:
print("Nom de la classe :", MaClasse.__name__)

#### `__bases__` — Classes parentes

In [ ]:
print("Classes parent de MaClasse :", MaClasse.__bases__)

#### `__class__` — Classe de l'objet

In [ ]:
objet = MaClasse()
print("Classe de l'objet :", objet.__class__)

#### `__dict__` — Attributs et méthodes

In [ ]:
print("Méthodes et attributs disponibles :", objet.__dict__)

#### `__doc__` — Documentation

In [ ]:
print("Documentation de la classe :", MaClasse.__doc__)

---

## Méthodes spéciales (Magic Methods)

Les méthodes spéciales permettent de **redéfinir le comportement** d'une classe pour les opérations courantes :

| Méthode | Description | Déclenchée par |
|---|---|---|
| `__init__` | Initialisation de l'objet | `MaClasse()` |
| `__repr__` | Représentation "officielle" (pour le débogage) | `repr(obj)` |
| `__str__` | Représentation "informelle" (pour l'utilisateur) | `str(obj)`, `print(obj)` |
| `__len__` | Renvoie la longueur de l'objet | `len(obj)` |
| `__getitem__` | Accès par index ou clé | `obj[key]` |

In [ ]:
class Voiture:
    _marque = "honda"
    _model  = "civic"
    _annee  = 2006
    
    def __repr__(self):
        return f"Voiture(marque={self._marque}, model={self._model}, annee={self._annee})"
    
    def __str__(self):
        return f"Voiture: {self._marque} {self._model} ({self._annee})"
    
    def __len__(self):
        # Definir la longueur de la voiture
        return len(self._marque) + len(self._model) + len(str(self._annee))
    
    def __getitem__(self, key):
        #permet d'acceder à un attribut specifique de la classe
        if key == "marque":
            return self._marque
        elif key == "model":
            return self._model
        elif key == "annee":
            return self._annee
        else:
            raise KeyError(f"Attribut {key} non trouvé")

### Exemple : utilisation des méthodes spéciales

In [ ]:
voiture = Voiture()

#### `__repr__` et `__str__`

In [ ]:
print(repr(voiture))  # Appelle __repr__
print(str(voiture))   # Appelle __str__

#### `__len__`

In [ ]:
print(len(voiture))  # Appelle __len__

#### `__getitem__`

In [ ]:
print(voiture["marque"])  # Appelle __getitem__
print(voiture["model"])   # Appelle __getitem__
print(voiture["annee"])   # Appelle __getitem__

---

## Opérateurs de comparaison (surcharge)

On peut redéfinir le comportement des opérateurs de comparaison pour nos objets :

| Méthode | Opérateur | Description |
|---|---|---|
| `__eq__` | `==` | Égalité |
| `__ne__` | `!=` | Différence |
| `__lt__` | `<` | Inférieur à |
| `__le__` | `<=` | Inférieur ou égal |
| `__gt__` | `>` | Supérieur à |
| `__ge__` | `>=` | Supérieur ou égal |

> 💡 En Python 3, `__ne__` (différent de) est **déduit automatiquement** de `__eq__` : si vous définissez `__eq__`, l'opérateur `!=` renvoie l'inverse sans que vous ayez besoin d'écrire `__ne__` à la main.

In [ ]:
class Car:
    _marque = ""
    _model  = ""
    _annee  = ""
    
    def __eq__(self, other):
        return self._marque == other._marque and self._model == other._model and self._annee == other._annee
    
    def __ne__(self, other):
        return not self.__eq__(other)
    
    def __lt__(self, other):
        if self._annee != other._annee:
            return self._annee < other._annee
        elif self._marque != other._marque:
            return self._marque < other._marque
        else:
            return self._model < other._model
        
    def __le__(self, other):
        return self < other or self == other
    
    def __gt__(self, other):
        return not (self < other or self == other)
    
    def __ge__(self, other):
        return not (self < other)

### Exemple de comparaison

In [ ]:
car1 = Car()
car1._marque = "Toyota"
car1._model = "Camry"
car1._annee = 2020

car2 = Car()
car2._marque = "Honda"
car2._model = "Accord"
car2._annee = 2019

car3 = Car()
car3._marque = "Toyota"
car3._model = "Corolla"
car3._annee = 2020

#### Test des comparaisons

In [ ]:
print(car1 == car2)  # False
print(car1 != car2)  # True
print(car1 < car2)   # False
print(car1 > car2)   # True
print(car1 <= car2)  # False
print(car1 >= car2)  # True

---

#### ⚠️ Piège : `__eq__` sans `__hash__`

Dès qu'une classe définit `__eq__` sans définir `__hash__`, ses objets deviennent **non hachables** : impossible de les mettre dans un `set` ou de les utiliser comme clés de dictionnaire (`TypeError`).

**Pourquoi ?** Un `set` range chaque objet dans un « casier » calculé à partir de son hash. Si on redéfinit l'égalité sans redéfinir le hash de façon cohérente, Python ne sait plus dans quel casier ranger l'objet — il préfère donc l'interdire.

**Le remède** : définir un `__hash__` cohérent avec `__eq__` (deux objets égaux doivent avoir le même hash), comme le fait la classe `Vecteur` plus bas avec `hash((self.x, self.y))`.

In [ ]:
# Car définit __eq__ mais pas __hash__ → ses objets ne peuvent plus aller dans un set
try:
    garage = {car1, car2}
except TypeError as e:
    print("TypeError :", e)

---

## Référence complète des méthodes spéciales (Dunder Methods)

### Cycle de vie
| Méthode | Déclenchée par | Description |
|---|---|---|
| `__new__(cls, ...)` | Avant `__init__` | Crée et retourne l'instance (rarement surchargée) |
| `__init__(self, ...)` | `MaClasse(...)` | Initialise l'instance |
| `__del__(self)` | Garbage collector | Appelée avant destruction (non fiable) |

### Représentation
| Méthode | Déclenchée par | Description |
|---|---|---|
| `__repr__(self)` | `repr(obj)` | Représentation officielle (debug) — doit idéalement permettre de recréer l'objet |
| `__str__(self)` | `str(obj)`, `print(obj)` | Représentation lisible pour l'utilisateur |
| `__format__(self, spec)` | `format(obj, spec)`, f-strings | Formatage personnalisé |
| `__bytes__(self)` | `bytes(obj)` | Représentation en bytes |

### Collections et itération
| Méthode | Déclenchée par | Description |
|---|---|---|
| `__len__(self)` | `len(obj)` | Longueur de l'objet |
| `__getitem__(self, key)` | `obj[key]` | Accès par index/clé |
| `__setitem__(self, key, val)` | `obj[key] = val` | Affectation par index/clé |
| `__delitem__(self, key)` | `del obj[key]` | Suppression par index/clé |
| `__contains__(self, item)` | `item in obj` | Test d'appartenance |
| `__iter__(self)` | `iter(obj)`, `for x in obj` | Retourne un itérateur |
| `__next__(self)` | `next(obj)` | Prochain élément de l'itérateur |
| `__reversed__(self)` | `reversed(obj)` | Itérateur inversé |

### Opérateurs arithmétiques
| Méthode | Opérateur | Méthode réfléchie |
|---|---|---|
| `__add__(self, other)` | `+` | `__radd__` |
| `__sub__(self, other)` | `-` | `__rsub__` |
| `__mul__(self, other)` | `*` | `__rmul__` |
| `__truediv__(self, other)` | `/` | `__rtruediv__` |
| `__floordiv__(self, other)` | `//` | `__rfloordiv__` |
| `__mod__(self, other)` | `%` | `__rmod__` |
| `__pow__(self, other)` | `**` | `__rpow__` |
| `__neg__(self)` | `-obj` (unaire) | — |
| `__abs__(self)` | `abs(obj)` | — |

### Opérateurs augmentés (`+=`, `-=`, ...)
| Méthode | Opérateur |
|---|---|
| `__iadd__(self, other)` | `+=` |
| `__isub__(self, other)` | `-=` |
| `__imul__(self, other)` | `*=` |

### Conversion de type
| Méthode | Déclenchée par |
|---|---|
| `__int__(self)` | `int(obj)` |
| `__float__(self)` | `float(obj)` |
| `__bool__(self)` | `bool(obj)`, `if obj:` |
| `__hash__(self)` | `hash(obj)`, utilisation dans dict/set |

### Contexte (`with`)
| Méthode | Déclenchée par |
|---|---|
| `__enter__(self)` | Entrée du bloc `with` |
| `__exit__(self, exc_type, exc_val, exc_tb)` | Sortie du bloc `with` |

In [ ]:
class Vecteur:
    """Exemple complet de magic methods sur un vecteur 2D."""

    def __init__(self, x, y):
        self.x = x
        self.y = y

    # --- Représentation ---
    def __repr__(self):
        return f"Vecteur({self.x}, {self.y})"

    def __str__(self):
        return f"({self.x}, {self.y})"

    # --- Arithmétique ---
    def __add__(self, other):
        return Vecteur(self.x + other.x, self.y + other.y)

    def __sub__(self, other):
        return Vecteur(self.x - other.x, self.y - other.y)

    def __mul__(self, scalaire):
        return Vecteur(self.x * scalaire, self.y * scalaire)

    def __rmul__(self, scalaire):  # pour 3 * vecteur
        return self.__mul__(scalaire)

    def __neg__(self):
        return Vecteur(-self.x, -self.y)

    def __abs__(self):
        return (self.x**2 + self.y**2) ** 0.5

    # --- Comparaison ---
    def __eq__(self, other):
        return self.x == other.x and self.y == other.y

    def __hash__(self):
        return hash((self.x, self.y))  # nécessaire si __eq__ est défini

    # --- Conversion ---
    def __bool__(self):
        return self.x != 0 or self.y != 0  # vecteur nul = False

    def __len__(self):
        return 2  # un vecteur 2D a toujours 2 composantes

    def __getitem__(self, index):
        if index == 0: return self.x
        if index == 1: return self.y
        raise IndexError("Index hors limites")

    def __iter__(self):
        yield self.x
        yield self.y


# Tests
v1 = Vecteur(1, 2)
v2 = Vecteur(3, 4)

print(repr(v1))           # Vecteur(1, 2)
print(str(v1))            # (1, 2)
print(v1 + v2)            # (4, 6)
print(v1 - v2)            # (-2, -2)
print(v1 * 3)             # (3, 6)
print(3 * v1)             # (3, 6)  — via __rmul__
print(-v1)                # (-1, -2)
print(abs(v2))            # 5.0
print(v1 == Vecteur(1,2)) # True
print(bool(Vecteur(0,0))) # False
print(list(v1))           # [1, 2]  — via __iter__
x, y = v1                 # unpacking — via __iter__

In [ ]:
# Gestionnaire de contexte (with) via __enter__ / __exit__
class GestionFichier:
    def __init__(self, nom, mode):
        self.nom = nom
        self.mode = mode
        self.fichier = None

    def __enter__(self):
        print(f"Ouverture de {self.nom}")
        self.fichier = open(self.nom, self.mode, encoding='utf-8')
        return self.fichier   # valeur assignée par 'as'

    def __exit__(self, exc_type, exc_val, exc_tb):
        print(f"Fermeture de {self.nom}")
        if self.fichier:
            self.fichier.close()
        # Retourner True supprime l'exception
        # Retourner False (ou None) la propage
        return False


# Usage
# with GestionFichier('test.txt', 'w') as f:
#     f.write('Hello!')
# Le fichier est automatiquement fermé, même en cas d'erreur